Neste  projeto sera feito  toda lipeza das base de dados, depois cronstrução tabelas fatos e dimensaos.

In [34]:
#Biblitecas
import re
import pandas as pd

In [3]:
# 1. Carregar a base de dados suja
# pd.read_excel lê o arquivo Excel original que contém inconsistências e erros de digitação.
df_raw = pd.read_excel('Base_Consultorio_Nutricionista_2024_2025_SUJA.xlsx')

In [35]:
# Criamos uma cópia de trabalho para preservar os dados originais
df = df_raw.copy()

In [ ]:
#2.  REMOÇÃO DE DUPLICADAS E VALORES NULOS ESSENCIAIS

# Removemos registros completamente duplicados que causam ruído na análise
df = df.drop_duplicates()

In [38]:
# 3. TRATAMENTO DE TEXTOS E PONTUAÇÃO (Nomes, Nutricionistas, Formas de Pgto, Status)


# Função auxiliar para limpar títulos médicos/acadêmicos dos nomes (ex: Dra., Dr., Nutri)
def limpar_titulo_nome(nome):
    if pd.isna(nome):
        return nome
    nome = str(nome).strip()
    # Remove prefixos como "Dra.", "Dr.", "Nutri", "Dra ", "Dr " de forma insensível ao caso
    nome = re.sub(
        r"^(Dra?\.|Dra|Dr|Nutri)\s+", "", nome, flags=re.IGNORECASE
    ).strip()
    return nome.title()  # Formata para 'Primeira Letra Maiúscula'


# Padronização da coluna de Nutricionistas
if "nutricionista" in df.columns:
    df["nutricionista"] = df["nutricionista"].apply(limpar_titulo_nome)

# Padronização do Nome do Paciente
if "nome_paciente" in df.columns:
    df["nome_paciente"] = df["nome_paciente"].apply(limpar_titulo_nome)

# Padronização da Forma de Pagamento (ex: 'pix', 'PIX', 'Pix ' -> 'Pix')
if "forma_pagamento" in df.columns:
    df["forma_pagamento"] = (
        df["forma_pagamento"].astype(str).str.strip().str.upper()
    )

# Padronização do Status da Consulta (ex: 'REALIZADA', 'realizada' -> 'Realizada')
if "status_consulta" in df.columns:
    df["status_consulta"] = (
        df["status_consulta"].astype(str).str.strip().str.title()
    )

# Padronização do Bairro/Localidade
if "bairro" in df.columns:
    df["bairro"] = df["bairro"].astype(str).str.strip().str.title()

In [39]:
# 4. TRATAMENTO DE DATAS (Datas inválidas como 31/02/2025)

# O parâmetro errors='coerce' converte datas impossíveis (ex: 31/02/2025) para NaT (Not a Time)
if "data_consulta" in df.columns:
    df["data_consulta"] = pd.to_datetime(
        df["data_consulta"], errors="coerce", format="%d/%m/%Y"
    )

In [40]:
# 5. TRATAMENTO DE VALORES MONETÁRIOS (ex: 'R$ 200,00', '200.00', '200')


def limpar_valor_monetario(valor):
    if pd.isna(valor):
        return None
    val_str = str(valor).strip()
    # Remove o prefixo R$, espaços e caracteres não numéricos exceto vírgula e ponto
    val_str = re.sub(r"[^\d.,]", "", val_str)
    # Se houver vírgula como separador decimal (ex: 200,00), substitui por ponto
    if "," in val_str and "." in val_str:
        val_str = val_str.replace(".", "").replace(",", ".")
    elif "," in val_str:
        val_str = val_str.replace(",", ".")
    try:
        return float(val_str)
    except ValueError:
        return None


if "valor_consulta" in df.columns:
    df["valor_consulta"] = df["valor_consulta"].apply(limpar_valor_monetario)

In [41]:
# 6. TRATAMENTO DE PESO E ALTURA (ex: '80,5 kg', '500', '1,75', '1.75m', '170cm')



# Tratamento do Peso (Padronização para kg)
def limpar_peso(peso):
    if pd.isna(peso):
        return None
    peso_str = str(peso).lower().replace(",", ".").strip()
    # Extrai apenas números e ponto
    numeros = re.findall(r"\d+\.?\d*", peso_str)
    if not numeros:
        return None
    val = float(numeros[0])
    # Trata possíveis discrepâncias/outliers extremos de digitação (ex: 500kg -> erro/divisão por 10)
    if val > 250:
        val = val / 10  # Ajuste de digitação comum (ex: 800 -> 80.0)
    return round(val, 2)


# Tratamento da Altura (Padronização para metros, ex: 1.75m)
def limpar_altura(altura):
    if pd.isna(altura):
        return None
    alt_str = str(altura).lower().replace(",", ".").strip()
    numeros = re.findall(r"\d+\.?\d*", alt_str)
    if not numeros:
        return None
    val = float(numeros[0])
    # Se estiver em centímetros (ex: 170cm ou 175), converte para metros (1.70 / 1.75)
    if val > 3.0:
        val = val / 100
    return round(val, 2)


if "peso" in df.columns:
    df["peso"] = df["peso"].apply(limpar_peso)

if "altura" in df.columns:
    df["altura"] = df["altura"].apply(limpar_altura)

In [43]:
# 7. MODELAGEM DIMENSIONAL (STAR SCHEMA)


# --- Dimensão: dim_paciente ---
cols_paciente = [
    "nome_paciente",
    "cpf",
    "telefone",
    "email",
    "genero",
    "peso",
    "altura",
]
dim_paciente = df[cols_paciente].drop_duplicates().reset_index(drop=True)
dim_paciente["id_paciente"] = dim_paciente.index + 1
# Reordenando colunas
dim_paciente = dim_paciente[["id_paciente"] + cols_paciente]

# --- Dimensão: dim_localidade ---
dim_localidade = (
    df[["bairro"]].drop_duplicates().dropna().reset_index(drop=True)
)
dim_localidade["id_bairro"] = dim_localidade.index + 1
dim_localidade = dim_localidade[["id_bairro", "bairro"]]

# --- Dimensão: dim_forma_pagamento ---
dim_forma_pagamento = (
    df[["forma_pagamento"]].drop_duplicates().dropna().reset_index(drop=True)
)
dim_forma_pagamento["id_forma_pagamento"] = dim_forma_pagamento.index + 1
dim_forma_pagamento = dim_forma_pagamento[
    ["id_forma_pagamento", "forma_pagamento"]
]

# --- Dimensão: dim_status ---
dim_status = (
    df[["status_consulta"]].drop_duplicates().dropna().reset_index(drop=True)
)
dim_status["id_status_consulta"] = dim_status.index + 1
dim_status = dim_status[["id_status_consulta", "status_consulta"]]

# --- Dimensão: dim_nutricionistas ---
dim_nutricionistas = (
    df[["nutricionista"]].drop_duplicates().dropna().reset_index(drop=True)
)
dim_nutricionistas["Id_nutricionista"] = dim_nutricionistas.index + 1

# Seleção reordenada das colunas com colchetes duplos:
dim_nutricionistas = dim_nutricionistas[["Id_nutricionista", "nutricionista"]]

In [ ]:

# CONSTRUÇÃO DA TABELA FATO (f_Atendimento)

f_Atendimento = df.copy()

# Merges com verificação de segurança
if 'id_paciente' in dim_paciente.columns:
    chaves = [c for c in ['nome_paciente', 'cpf'] if c in f_Atendimento.columns and c in dim_paciente.columns]
    f_Atendimento = f_Atendimento.merge(dim_paciente[['id_paciente'] + chaves], on=chaves, how='left')

if 'id_bairro' in dim_localidade.columns and 'bairro' in f_Atendimento.columns:
    f_Atendimento = f_Atendimento.merge(dim_localidade[['id_bairro', 'bairro']], on='bairro', how='left')

if 'id_forma_pagamento' in dim_forma_pagamento.columns and 'forma_pagamento' in f_Atendimento.columns:
    f_Atendimento = f_Atendimento.merge(dim_forma_pagamento[['id_forma_pagamento', 'forma_pagamento']], on='forma_pagamento', how='left')

if 'id_status_consulta' in dim_status.columns and 'status_consulta' in f_Atendimento.columns:
    f_Atendimento = f_Atendimento.merge(dim_status[['id_status_consulta', 'status_consulta']], on='status_consulta', how='left')

if 'Id_nutricionista' in dim_nutricionistas.columns and 'nutricionista' in f_Atendimento.columns:
    f_Atendimento = f_Atendimento.merge(dim_nutricionistas[['Id_nutricionista', 'nutricionista']], on='nutricionista', how='left')

# Id da consulta (Chave Primária da Fato)
f_Atendimento['id_consulta'] = f_Atendimento.index + 1

# Seleção segura das colunas
colunas_fato = [
    "id_consulta", "id_paciente", "id_bairro", 
    "id_forma_pagamento", "id_status_consulta", 
    "Id_nutricionista", "data_consulta", "valor_consulta", "objetivo"
]

colunas_validas = [col for col in colunas_fato if col in f_Atendimento.columns]
f_Atendimento = f_Atendimento[colunas_validas]

In [46]:
# 9. EXPORTAÇÃO DAS TABELAS FINALIZADAS (EXCEL / CSV)

with pd.ExcelWriter("Base_Tratada_Modelada_Nutricao.xlsx") as writer:
    f_Atendimento.to_excel(writer, sheet_name="f_Atendimento", index=False)
    dim_paciente.to_excel(writer, sheet_name="dim_paciente", index=False)
    dim_localidade.to_excel(writer, sheet_name="dim_localidade", index=False)
    dim_forma_pagamento.to_excel(
        writer, sheet_name="dim_forma_pagamento", index=False
    )
    dim_status.to_excel(writer, sheet_name="dim_status", index=False)
    dim_nutricionistas.to_excel(
        writer, sheet_name="dim_nutricionistas", index=False
    )

print(
    "Tratamento e Modelagem Dimensional concluídos com sucesso! Arquivo gerado."
)

Tratamento e Modelagem Dimensional concluídos com sucesso! Arquivo gerado.
